# 4.0 Optimize TP/SL Model Packs

This notebook combines the training flow and production-parity backtest flow to optimize `(TP_MULT, SL_MULT)` for model-pack selection.

## Locked Decisions
- Bayesian optimizer: Optuna
- Search space: `TP_MULT in [1.5, 5.0]`, `SL_MULT in [1.0, 3.5]`
- Budget: `40` Bayesian trials + `25` fallback grid points
- Objective: `score = net_usd + 100 * sharpe_ratio`
- Full retraining per TP/SL candidate
- Holdout + production-style backtest during optimization

Run all cells top-to-bottom.

In [6]:
import os
import json
import inspect
import pickle
import warnings
from pathlib import Path
from itertools import product
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import plotly.express as px

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from Learn.features import _add_features_US500_v2
from Learn.labels import calculate_trade_outcomes_capped
from Learn.preprocess import preprocess_ohlcv

warnings.filterwarnings('ignore')

try:
    import talib
except Exception as exc:
    raise ImportError('Missing talib. Install with: pip install TA-Lib') from exc

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception:
    OPTUNA_AVAILABLE = False
    print('Optuna not installed. Bayesian phase will be skipped and grid fallback will run.')

print('Setup complete.')

Setup complete.


In [7]:
# =========================
# Configuration
# =========================

# Data
DS_NAME = '../data/US500_M1_520weeks.csv'
N_ROWS = 500_000
TRAIN_FRACTION = 0.90
WARMUP_BARS = 7_000

# Search
USE_BAYESIAN = True
N_BAYES_TRIALS = 40
TP_RANGE = (1.5, 5.0)
SL_RANGE = (1.0, 3.5)
GRID_TP_VALUES = [1.5, 2.375, 3.25, 4.125, 5.0]
GRID_SL_VALUES = [1.0, 1.625, 2.25, 2.875, 3.5]

# Optional smoke mode for quick validation
SMOKE_MODE = False
SMOKE_BAYES_TRIALS = 3
SMOKE_GRID_LIMIT = 4

# Optimization constraints
OPTIMIZE_ONLY_TP_SL = True
VALIDATION_MODE = 'holdout_plus_backtest'

# Targets and model config
TARGET_COLS = ['long_quality', 'short_quality', 'signed_quality']
PRIMARY_TARGET = 'signed_quality'
TARGET_CLIP_MAX_MULT = 1.5

LGBM_BASE_PARAMS = {
    'objective': 'regression',
    'boosting_type': 'gbdt',
    'n_estimators': 1200,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'min_child_samples': 80,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

# Production-emulation signal controls
TRADE_THRESHOLD = 0.80
QUALITY_THRESHOLD = 0.60
PATIENCE = 1
ENTRY_BUFFER = 0.00001
MAX_POS_SIZE = None
TRADING_HOURS = None

# Risk / evaluation controls
RISK = 10.0
COMMISSION_USD = 0.00
LOOKAHEAD_BARS = 30

# Labeling controls
OUTCOME_BASE_PARAMS = {
    'atr_window': 60,
    'max_horizon': 30,
}
REGIME_PARAMS = {
    'ma_period': 90,
    'slope_smoothness': 50,
    'regime_min_duration': 0,
    'atr_window': 60,
    'atr_lookback': 720,
    'atr_percentile': 0.0,
    'slope_threshold': 0,
}

# Ranking
ALPHA_SHARPE = 100
MIN_TRADES_BASE = 25
MIN_TRADES_FRAC = 0.003

# Paths
MODEL_PACK_DIR = Path('ModelPacks')
MODEL_PACK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path('.')

assert os.path.exists(DS_NAME), f'Dataset not found: {DS_NAME}'
print('Config ready.')

Config ready.


In [8]:
# =========================
# Shared utility functions
# =========================

def load_ohlcv(ds_name, n_rows=None):
    df = pd.read_csv(ds_name)
    if n_rows is not None:
        df = df.tail(n_rows)
    df = df.sort_values('Time').reset_index(drop=True)
    df['Time'] = pd.to_datetime(df['Time'])
    return df


def FEATURES(df, include_mtf=True, regime_params=None, selected_features=None):
    kwargs = {'include_mtf': include_mtf}
    if regime_params is not None:
        kwargs['regime_params'] = regime_params
    if selected_features is not None:
        kwargs['selected_features'] = selected_features
    return _add_features_US500_v2(df, **kwargs)


def build_features(df_raw, regime_params, selected_features=None):
    df_feat = FEATURES(
        df_raw.copy(),
        include_mtf=True,
        regime_params=regime_params,
        selected_features=selected_features,
    )
    df_feat = df_feat.dropna().reset_index(drop=True)
    return df_feat


def safe_spearman(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.nanstd(y_true) == 0.0 or np.nanstd(y_pred) == 0.0:
        return 0.0
    result = stats.spearmanr(y_true, y_pred, nan_policy='omit')
    value = getattr(result, 'statistic', np.nan)
    if not np.isfinite(value):
        return 0.0
    return float(value)


def reg_metrics(y_true, y_pred):
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': float(r2_score(y_true, y_pred)),
        'Spearman': float(safe_spearman(y_true, y_pred)),
    }


def calc_sharpe(returns, annualization=None):
    arr = np.asarray(returns, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size < 2:
        return 0.0
    std = float(arr.std(ddof=1))
    if std <= 1e-12:
        return 0.0
    sr = float(arr.mean()) / std
    if annualization is not None and float(annualization) > 0:
        sr = sr * np.sqrt(float(annualization))
    return float(sr)


def calc_max_drawdown(equity_curve):
    eq = np.asarray(equity_curve, dtype=float)
    if eq.size == 0:
        return 0.0
    run_max = np.maximum.accumulate(eq)
    dd = eq - run_max
    return float(dd.min())


def compute_targets(df_feat, tp_mult, sl_mult, outcome_base_params, clip_max=None):
    params = dict(outcome_base_params)
    params['tp_mult'] = float(tp_mult)
    params['sl_mult'] = float(sl_mult)

    if 'atr' not in df_feat.columns:
        df_feat['atr'] = talib.ATR(
            df_feat['High'].values.astype(float),
            df_feat['Low'].values.astype(float),
            df_feat['Close'].values.astype(float),
            timeperiod=14,
        )

    outcomes = calculate_trade_outcomes_capped(df_feat, **params)
    eps = 1e-8
    long_q = np.log1p(outcomes['buy_MFE'] / (outcomes['buy_MAE'] + eps))
    short_q = np.log1p(outcomes['sell_MFE'] / (outcomes['sell_MAE'] + eps))

    if clip_max is not None:
        long_q = np.clip(long_q, 0.0, clip_max)
        short_q = np.clip(short_q, 0.0, clip_max)

    out = df_feat.copy()
    out['long_quality'] = np.asarray(long_q, dtype=float)
    out['short_quality'] = np.asarray(short_q, dtype=float)
    out['signed_quality'] = out['long_quality'] - out['short_quality']
    out = out.dropna().reset_index(drop=True)
    return out, outcomes, params


def rounded_pair(tp_mult, sl_mult, ndigits=4):
    return (round(float(tp_mult), ndigits), round(float(sl_mult), ndigits))


print('Shared utilities ready.')

Shared utilities ready.


In [9]:
# =========================
# Training utility layer
# =========================

def export_pack(pack, pack_path):
    with open(pack_path, 'wb') as fh:
        pickle.dump(pack, fh)


def train_pack_for_tp_sl(tp_mult, sl_mult, run_tag='opt', selected_features=None):
    tp_mult = float(tp_mult)
    sl_mult = float(sl_mult)

    df_raw = load_ohlcv(DS_NAME, n_rows=N_ROWS)
    df_feat = build_features(df_raw, regime_params=REGIME_PARAMS, selected_features=selected_features)

    clip_max = max(tp_mult, sl_mult) * TARGET_CLIP_MAX_MULT
    df_lab, _, outcome_params = compute_targets(
        df_feat, tp_mult=tp_mult, sl_mult=sl_mult, outcome_base_params=OUTCOME_BASE_PARAMS, clip_max=clip_max
    )

    split_idx = int(len(df_lab) * TRAIN_FRACTION)
    df_train = df_lab.iloc[:split_idx].copy().reset_index(drop=True)
    df_holdout = df_lab.iloc[split_idx:].copy().reset_index(drop=True)

    preprocess_args = {
        'target_col': TARGET_COLS,
        'outcomes_col': None,
        'shift': 0,
        'onehot_prefixes': ['OH_'],
        'price_prefixes': ['PR_'],
    }

    X_train, y_train, scaler, features, _, _ = preprocess_ohlcv(
        df_train, **preprocess_args, scaler=None, return_df=True
    )
    X_holdout, y_holdout, _, _, _, _ = preprocess_ohlcv(
        df_holdout, **preprocess_args, scaler=scaler, return_df=True
    )

    model_params = {target: dict(LGBM_BASE_PARAMS) for target in TARGET_COLS}
    models = {}
    holdout_metrics = {}

    for i, target in enumerate(TARGET_COLS):
        model = LGBMRegressor(**model_params[target])
        model.fit(X_train, y_train[:, i])
        pred_holdout = model.predict(X_holdout)
        models[target] = model
        holdout_metrics[target] = reg_metrics(y_holdout[:, i], pred_holdout)

    today = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    ds_title = Path(DS_NAME).stem
    model_name = f'{ds_title}_LightGBM_{run_tag}_TP{tp_mult:.4f}_SL{sl_mult:.4f}_{today}_model.pkl'
    pack_path = MODEL_PACK_DIR / model_name

    pack = {
        'model': models[PRIMARY_TARGET],
        'primary_model': models[PRIMARY_TARGET],
        'aux_models': {k: v for k, v in models.items() if k != PRIMARY_TARGET},
        'model_info': {
            'dataset_name': ds_title,
            'dataset_dir': DS_NAME,
            'date_trained': today,
            'model_type': 'LightGBMRegressionPack',
            'task': 'regression',
            'primary_target': PRIMARY_TARGET,
            'targets': TARGET_COLS,
        },
        'model_params': model_params,
        'features': list(features),
        'selected_features': list(features),
        'feature_function': FEATURES,
        'feature_function_source': inspect.getsource(FEATURES),
        'preprocess_function': preprocess_ohlcv,
        'preprocess_function_source': inspect.getsource(preprocess_ohlcv),
        'preprocess_args': preprocess_args,
        'scaler': scaler,
        'regime_params': dict(REGIME_PARAMS),
        'outcome_params': dict(outcome_params),
        'trading_hours': TRADING_HOURS,
        'feature_count': int(X_train.shape[1]),
        'validation': {'holdout': holdout_metrics},
    }

    export_pack(pack, pack_path)

    return {
        'pack_path': str(pack_path),
        'tp_mult': tp_mult,
        'sl_mult': sl_mult,
        'feature_count': int(X_train.shape[1]),
        'holdout_metrics': holdout_metrics,
        'rows_train': int(len(df_train)),
        'rows_holdout': int(len(df_holdout)),
    }


print('Training utilities ready.')

Training utilities ready.


In [ ]:
# =========================
# Backtest utility layer
# =========================

def load_pack(pack_path):
    with open(pack_path, 'rb') as f:
        return pickle.load(f)


def predict_pack_outputs(pack, df_raw_eval):
    primary_model = pack.get('primary_model', pack.get('model'))
    aux_models = pack.get('aux_models', {})
    selected_features = pack.get('selected_features', None)
    # If selected_features contains preprocessed names (e.g. OH_/PR_),
    # it is not valid input for raw feature generation.
    if isinstance(selected_features, list) and any(str(c).startswith(('OH_', 'PR_')) for c in selected_features):
        selected_features = None
    regime_params = pack.get('regime_params', None)
    preprocess_fn = pack['preprocess_function']
    preprocess_args = dict(pack['preprocess_args'])
    scaler = pack.get('scaler')
    features = pack.get('features', [])

    df_feat = build_features(df_raw_eval, regime_params=regime_params, selected_features=selected_features)

    # Compatibility patch: older/newer feature builders may omit ATR while
    # pack feature schemas can still include it.
    if ('atr' in features) and ('atr' not in df_feat.columns):
        df_feat['atr'] = talib.ATR(
            df_feat['High'].values.astype(float),
            df_feat['Low'].values.astype(float),
            df_feat['Close'].values.astype(float),
            timeperiod=14,
        )

    missing = [c for c in features if c not in df_feat.columns]
    if missing:
        raise ValueError(f'Missing features for inference: {missing[:10]}')

    keep_cols = [c for c in ['Time', 'Open', 'High', 'Low', 'Close', 'Volume'] + features if c in df_feat.columns]
    df_feat = df_feat[keep_cols]

    prep_eval_args = dict(preprocess_args)
    prep_eval_args['target_col'] = None
    prep_eval_args['outcomes_col'] = None

    _, _, _, _, _, df_eval_proc = preprocess_fn(
        df_feat, scaler=scaler, return_df=True, **prep_eval_args
    )

    # Build inference matrix from processed frame and enforce model expected width.
    expected_n = int(getattr(primary_model, 'n_features_in_', 0) or 0)
    feature_cols = list(features) if isinstance(features, (list, tuple)) else []
    if not feature_cols:
        feature_cols = [c for c in df_eval_proc.columns if c not in ['Time']]

    # Keep only known processed columns and preserve order.
    feature_cols = [c for c in feature_cols if c in df_eval_proc.columns]
    X_eval_df = df_eval_proc[feature_cols].copy()

    if expected_n > 0:
        if X_eval_df.shape[1] < expected_n:
            raise ValueError(
                f'Inference matrix has {X_eval_df.shape[1]} cols, but model expects {expected_n}'
            )
        if X_eval_df.shape[1] > expected_n:
            # Some packs carry a wider schema than the trained model;
            # use the left-aligned trained-width slice for compatibility.
            X_eval_df = X_eval_df.iloc[:, :expected_n]
    X_eval = np.ascontiguousarray(X_eval_df.values)

    if isinstance(aux_models, dict) and {'long_quality', 'short_quality'}.issubset(aux_models.keys()):
        pred_long = np.asarray(aux_models['long_quality'].predict(X_eval)).reshape(-1)
        pred_short = np.asarray(aux_models['short_quality'].predict(X_eval)).reshape(-1)
    else:
        pred_signed = np.asarray(primary_model.predict(X_eval)).reshape(-1)
        pred_long = np.maximum(pred_signed, 0.0)
        pred_short = np.maximum(-pred_signed, 0.0)

    pred_signed_direct = np.asarray(primary_model.predict(X_eval)).reshape(-1)
    pred_signed = pred_long - pred_short

    pred_df = pd.DataFrame({
        'Time': pd.to_datetime(df_eval_proc['Time']).reset_index(drop=True),
        'pred_long': pred_long,
        'pred_short': pred_short,
        'pred_signed': pred_signed,
        'pred_signed_direct': pred_signed_direct,
    })
    return pred_df


def prediction_to_signal(row, trade_threshold, quality_threshold, trading_hours):
    if trading_hours is not None:
        t_start = pd.to_datetime(trading_hours[0]).time()
        t_end = pd.to_datetime(trading_hours[1]).time()
        in_restricted_hours = t_start <= row['Time'].time() <= t_end
    else:
        in_restricted_hours = False

    quality = abs(float(row['pred_signed']))
    buy_ready = (float(row['pred_long']) >= float(trade_threshold)) and (quality >= float(quality_threshold))
    sell_ready = (float(row['pred_short']) >= float(trade_threshold)) and (quality >= float(quality_threshold))

    if in_restricted_hours:
        signal = 0
    elif buy_ready:
        signal = 1
    elif sell_ready:
        signal = -1
    else:
        signal = 0

    return pd.Series({
        'quality': quality,
        'raw_signal': int(signal),
    })


def simulate_production_trades(signal_frame, *, atr_window, tp_mult, sl_mult, patience, entry_buffer, trading_hours,
                               risk_per_trade=RISK, commission_per_lot=COMMISSION_USD, max_pos_size=None):
    df_sim = signal_frame.copy().reset_index(drop=True)
    df_sim['atr'] = talib.ATR(
        df_sim['High'].values.astype(float),
        df_sim['Low'].values.astype(float),
        df_sim['Close'].values.astype(float),
        timeperiod=int(atr_window),
    )

    trades = []
    pending_order = None
    open_position = None
    countdown = 0

    for i, row in df_sim.iterrows():
        bar_time = row['Time']
        bar_open = float(row['Open'])
        bar_high = float(row['High'])
        bar_low = float(row['Low'])

        had_pending = pending_order is not None
        had_open = open_position is not None

        if countdown > 0:
            countdown -= 1

        if pending_order is not None:
            side = pending_order['side']
            entry = pending_order['entry']
            if side == 'buy' and bar_high >= entry:
                fill_price = bar_open if bar_open >= entry else entry
                open_position = {**pending_order, 'fill_time': bar_time, 'fill_price': fill_price, 'entry_index': i}
                pending_order = None
            elif side == 'sell' and bar_low <= entry:
                fill_price = bar_open if bar_open <= entry else entry
                open_position = {**pending_order, 'fill_time': bar_time, 'fill_price': fill_price, 'entry_index': i}
                pending_order = None

        if pending_order is not None and bar_time >= pending_order['expires_at']:
            pending_order = None

        if open_position is not None:
            side = open_position['side']
            fill_price = float(open_position['fill_price'])
            sl = float(open_position['sl'])
            tp = float(open_position['tp'])
            position_size = float(open_position['position_size'])
            close_price = None
            close_reason = None

            if side == 'buy':
                sl_hit = sl > 0 and bar_low <= sl
                tp_hit = tp > 0 and bar_high >= tp
                if sl_hit:
                    close_price = min(sl, bar_open)
                    close_reason = 'sl'
                elif tp_hit:
                    close_price = tp
                    close_reason = 'tp'
            else:
                sl_hit = sl > 0 and bar_high >= sl
                tp_hit = tp > 0 and bar_low <= tp
                if sl_hit:
                    close_price = max(sl, bar_open)
                    close_reason = 'sl'
                elif tp_hit:
                    close_price = tp
                    close_reason = 'tp'

            if close_price is not None:
                if close_reason == 'tp':
                    raw_usd = risk_per_trade * (tp_mult / sl_mult)
                elif close_reason == 'sl':
                    raw_usd = -risk_per_trade
                else:
                    raw_usd = (close_price - fill_price) * position_size * 100_000 if side == 'buy' else (fill_price - close_price) * position_size * 100_000

                commission = position_size * commission_per_lot
                net_usd = raw_usd - commission
                trades.append({
                    'signal_time': open_position['signal_time'],
                    'fill_time': open_position['fill_time'],
                    'exit_time': bar_time,
                    'direction': 'BUY' if side == 'buy' else 'SELL',
                    'outcome': close_reason,
                    'position_size': position_size,
                    'raw_usd': raw_usd,
                    'commission': commission,
                    'net_usd': net_usd,
                })
                open_position = None

        if had_pending or had_open or countdown > 0:
            continue

        if trading_hours is not None:
            t_start = pd.to_datetime(trading_hours[0]).time()
            t_end = pd.to_datetime(trading_hours[1]).time()
            if t_start <= row['Time'].time() <= t_end:
                continue

        if pd.isna(row['atr']):
            continue

        signal = int(row['raw_signal'])
        if signal == 0:
            continue

        if signal == 1:
            side = 'buy'
            entry = float(row['High']) + entry_buffer
            tp = float(row['High']) + (tp_mult * float(row['atr']))
            sl = float(row['High']) - (sl_mult * float(row['atr']))
        else:
            side = 'sell'
            entry = float(row['Low']) - entry_buffer
            tp = float(row['Low']) - (tp_mult * float(row['atr']))
            sl = float(row['Low']) + (sl_mult * float(row['atr']))

        sl_distance = abs(entry - sl)
        position_size = (risk_per_trade / sl_distance) / 100_000.0 if sl_distance > 0 else 0.0

        if max_pos_size is not None and position_size > max_pos_size:
            continue

        pending_order = {
            'signal_time': row['Time'],
            'side': side,
            'entry': float(entry),
            'tp': float(tp),
            'sl': float(sl),
            'position_size': float(position_size),
            'expires_at': row['Time'] + pd.Timedelta(minutes=int(patience)),
        }
        countdown = int(patience)

    trades_df = pd.DataFrame(trades)
    return df_sim, trades_df


def evaluate_pack(pack_path, eval_df_raw, cfg):
    pack = load_pack(pack_path)

    # Pack values override runtime defaults where applicable
    trading_hours = pack.get('trading_hours', cfg['TRADING_HOURS'])
    patience = int(pack.get('patience', cfg['PATIENCE']))

    outcome_params = dict(pack.get('outcome_params', {}) or {})
    tp_mult = float(outcome_params.get('tp_mult'))
    sl_mult = float(outcome_params.get('sl_mult'))
    atr_window = int(outcome_params.get('atr_window', 14))

    pred_df = predict_pack_outputs(pack, eval_df_raw.copy())
    frame = eval_df_raw.merge(pred_df, on='Time', how='inner').sort_values('Time').reset_index(drop=True)

    signal_cols = frame.apply(
        prediction_to_signal,
        axis=1,
        trade_threshold=cfg['TRADE_THRESHOLD'],
        quality_threshold=cfg['QUALITY_THRESHOLD'],
        trading_hours=trading_hours,
    )
    frame = pd.concat([frame, signal_cols], axis=1)

    _, trades = simulate_production_trades(
        frame,
        atr_window=atr_window,
        tp_mult=tp_mult,
        sl_mult=sl_mult,
        patience=patience,
        entry_buffer=cfg['ENTRY_BUFFER'],
        trading_hours=trading_hours,
        risk_per_trade=cfg['RISK'],
        commission_per_lot=cfg['COMMISSION_USD'],
        max_pos_size=cfg['MAX_POS_SIZE'],
    )

    if len(trades) == 0:
        return {
            'pack_path': str(pack_path),
            'tp_mult': tp_mult,
            'sl_mult': sl_mult,
            'net_usd': 0.0,
            'sharpe_ratio': 0.0,
            'max_drawdown': 0.0,
            'trade_count': 0,
            'win_rate': 0.0,
            'avg_trade_pnl': 0.0,
            'equity_curve': [],
        }

    trades = trades.sort_values('exit_time').reset_index(drop=True)
    trades['cum_usd'] = trades['net_usd'].cumsum()

    net_usd = float(trades['net_usd'].sum())
    returns = trades['net_usd'].values / max(cfg['RISK'], 1e-8)
    sharpe_ratio = float(calc_sharpe(returns))
    max_drawdown = float(calc_max_drawdown(trades['cum_usd'].values))
    trade_count = int(len(trades))
    win_rate = float((trades['outcome'] == 'tp').mean())
    avg_trade_pnl = float(trades['net_usd'].mean())

    return {
        'pack_path': str(pack_path),
        'tp_mult': tp_mult,
        'sl_mult': sl_mult,
        'net_usd': net_usd,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'trade_count': trade_count,
        'win_rate': win_rate,
        'avg_trade_pnl': avg_trade_pnl,
        'equity_curve': trades['cum_usd'].astype(float).tolist(),
    }


eval_df_raw = load_ohlcv(DS_NAME, n_rows=N_ROWS)
print(f'Eval data loaded: {len(eval_df_raw):,} rows')

Eval data loaded: 500,000 rows


In [18]:
# Validation probe (fast): evaluate one existing trained pack without retraining
probe_pack_path = 'ModelPacks/US500_M1_520weeks_LightGBM_bayes_TP2.5567_SL3.0519_20260704_164653_model.pkl'
probe_metrics = evaluate_pack(probe_pack_path, eval_df_raw=eval_df_raw, cfg=cfg_eval)
probe_metrics['score'] = score_candidate(probe_metrics) if 'score_candidate' in globals() else (probe_metrics['net_usd'] + ALPHA_SHARPE * probe_metrics['sharpe_ratio'])
print('Probe metrics summary:')
print({k: probe_metrics[k] for k in ['tp_mult','sl_mult','trade_count','net_usd','sharpe_ratio','score']})

ValueError: X has 205 features, but LGBMRegressor is expecting 199 features as input.

In [11]:
# =========================
# Optimization: Bayesian phase
# =========================

candidate_cache = {}
results_rows = []
rejected_rows = []

min_trades_for_valid_score = max(MIN_TRADES_BASE, int(len(eval_df_raw) * MIN_TRADES_FRAC))
print('Min trades for valid score:', min_trades_for_valid_score)

cfg_eval = {
    'TRADE_THRESHOLD': TRADE_THRESHOLD,
    'QUALITY_THRESHOLD': QUALITY_THRESHOLD,
    'PATIENCE': PATIENCE,
    'ENTRY_BUFFER': ENTRY_BUFFER,
    'MAX_POS_SIZE': MAX_POS_SIZE,
    'TRADING_HOURS': TRADING_HOURS,
    'RISK': RISK,
    'COMMISSION_USD': COMMISSION_USD,
}


def score_candidate(metrics, alpha_sharpe=ALPHA_SHARPE):
    base_score = float(metrics['net_usd']) + float(alpha_sharpe) * float(metrics['sharpe_ratio'])
    tc = int(metrics['trade_count'])
    if tc < min_trades_for_valid_score:
        ratio = tc / max(min_trades_for_valid_score, 1)
        base_score = base_score * (0.5 + 0.5 * ratio)
    return float(base_score)


def evaluate_candidate(tp_mult, sl_mult, search_method):
    key = rounded_pair(tp_mult, sl_mult, ndigits=4)

    if key in candidate_cache:
        cached = dict(candidate_cache[key])
        cached['search_method'] = search_method
        return cached

    try:
        train_meta = train_pack_for_tp_sl(tp_mult=key[0], sl_mult=key[1], run_tag=search_method)
        metrics = evaluate_pack(train_meta['pack_path'], eval_df_raw=eval_df_raw, cfg=cfg_eval)

        metrics['score'] = score_candidate(metrics)
        metrics['search_method'] = search_method
        metrics['status'] = 'ok'
        metrics['feature_count'] = train_meta['feature_count']

        candidate_cache[key] = dict(metrics)
        return metrics

    except Exception as exc:
        failed = {
            'tp_mult': key[0],
            'sl_mult': key[1],
            'search_method': search_method,
            'status': 'failed',
            'error': str(exc),
        }
        rejected_rows.append(failed)
        return failed


if USE_BAYESIAN and OPTUNA_AVAILABLE:
    n_trials = SMOKE_BAYES_TRIALS if SMOKE_MODE else N_BAYES_TRIALS

    def objective(trial):
        tp_mult = trial.suggest_float('tp_mult', TP_RANGE[0], TP_RANGE[1])
        sl_mult = trial.suggest_float('sl_mult', SL_RANGE[0], SL_RANGE[1])
        out = evaluate_candidate(tp_mult, sl_mult, search_method='bayes')
        if out.get('status') != 'ok':
            return -1e12

        row = dict(out)
        row['trial_number'] = int(trial.number)
        results_rows.append(row)
        return float(out['score'])

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    print('Bayesian phase complete:', n_trials, 'trials')
else:
    print('Bayesian phase skipped (disabled or Optuna unavailable).')

[I 2026-07-04 16:39:18,221] A new study created in memory with name: no-name-c454f5aa-8505-4b09-a2c7-ed09f1967cb6


Min trades for valid score: 1500


[I 2026-07-04 16:47:58,267] Trial 0 finished with value: -1000000000000.0 and parameters: {'tp_mult': 2.556694517412185, 'sl_mult': 3.0518940153468694}. Best is trial 0 with value: -1000000000000.0.
[I 2026-07-04 16:53:56,288] Trial 1 finished with value: -1000000000000.0 and parameters: {'tp_mult': 2.5877347958033408, 'sl_mult': 1.3813645178737677}. Best is trial 0 with value: -1000000000000.0.
[I 2026-07-04 16:59:58,420] Trial 2 finished with value: -1000000000000.0 and parameters: {'tp_mult': 2.1889587729651794, 'sl_mult': 1.7294600659646968}. Best is trial 0 with value: -1000000000000.0.
[I 2026-07-04 17:06:05,646] Trial 3 finished with value: -1000000000000.0 and parameters: {'tp_mult': 2.627094058318991, 'sl_mult': 1.9222239484645158}. Best is trial 0 with value: -1000000000000.0.
[I 2026-07-04 17:11:59,949] Trial 4 finished with value: -1000000000000.0 and parameters: {'tp_mult': 4.856230388823645, 'sl_mult': 1.5622801545964409}. Best is trial 0 with value: -1000000000000.0.
[I 

KeyboardInterrupt: 

In [ ]:
# =========================
# Optimization: fallback grid
# =========================

grid_pairs = list(product(GRID_TP_VALUES, GRID_SL_VALUES))
if SMOKE_MODE:
    grid_pairs = grid_pairs[:SMOKE_GRID_LIMIT]

for tp_mult, sl_mult in grid_pairs:
    out = evaluate_candidate(tp_mult, sl_mult, search_method='grid')
    if out.get('status') == 'ok':
        results_rows.append(dict(out))

print('Grid phase complete:', len(grid_pairs), 'pairs')

In [ ]:
# =========================
# Merge, rank, and recommend
# =========================

results_df = pd.DataFrame(results_rows)
if results_df.empty:
    raise RuntimeError('No successful candidates were evaluated.')

# Deduplicate by TP/SL and keep best score per pair
results_df = results_df.sort_values('score', ascending=False).drop_duplicates(['tp_mult', 'sl_mult'], keep='first')

low_trade_mask = results_df['trade_count'] < min_trades_for_valid_score
if low_trade_mask.any():
    low_trade_rejected = results_df.loc[low_trade_mask, ['tp_mult', 'sl_mult', 'trade_count', 'score']].copy()
    low_trade_rejected['status'] = 'low_trade_count'
    rejected_rows.extend(low_trade_rejected.to_dict(orient='records'))

results_df = results_df.sort_values(
    ['score', 'net_usd', 'sharpe_ratio'],
    ascending=[False, False, False],
).reset_index(drop=True)

best_candidate = results_df.iloc[0].to_dict()
rejected_df = pd.DataFrame(rejected_rows)

print('Top 10 candidates:')
display(results_df.head(10)[[
    'tp_mult', 'sl_mult', 'search_method', 'score', 'net_usd',
    'sharpe_ratio', 'max_drawdown', 'trade_count', 'win_rate', 'avg_trade_pnl'
]])

print('Best candidate:')
display(pd.DataFrame([best_candidate])[['tp_mult', 'sl_mult', 'score', 'net_usd', 'sharpe_ratio', 'trade_count', 'pack_path']])

if not rejected_df.empty:
    print('Rejected candidates:')
    display(rejected_df.head(20))

In [ ]:
# =========================
# Diagnostics and plots
# =========================

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sc = ax.scatter(results_df['net_usd'], results_df['sharpe_ratio'], c=results_df['score'], cmap='viridis')
ax.set_title('Net Profit vs Sharpe (color=score)')
ax.set_xlabel('net_usd')
ax.set_ylabel('sharpe_ratio')
plt.colorbar(sc, ax=ax, label='score')
plt.show()

pivot = results_df.pivot_table(index='sl_mult', columns='tp_mult', values='score', aggfunc='max')
plt.figure(figsize=(8, 5))
plt.imshow(pivot.values, aspect='auto', origin='lower')
plt.xticks(ticks=np.arange(len(pivot.columns)), labels=[f'{v:.3f}' for v in pivot.columns], rotation=45)
plt.yticks(ticks=np.arange(len(pivot.index)), labels=[f'{v:.3f}' for v in pivot.index])
plt.title('TP/SL Score Heatmap')
plt.xlabel('TP_MULT')
plt.ylabel('SL_MULT')
plt.colorbar(label='score')
plt.tight_layout()
plt.show()

best_equity = best_candidate.get('equity_curve', [])
if isinstance(best_equity, list) and len(best_equity) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(best_equity)
    plt.title('Equity Curve - Best Candidate')
    plt.xlabel('Trade Index')
    plt.ylabel('Cumulative Net USD')
    plt.grid(alpha=0.3)
    plt.show()

# Optional interactive scatter
fig_px = px.scatter(
    results_df,
    x='net_usd',
    y='sharpe_ratio',
    color='score',
    hover_data=['tp_mult', 'sl_mult', 'trade_count', 'search_method'],
    title='Optimization Candidates: Net Profit vs Sharpe',
)
fig_px.show()

In [ ]:
# =========================
# Export artifacts
# =========================

results_csv_path = OUTPUT_DIR / 'tp_sl_optimization_results.csv'
results_json_path = OUTPUT_DIR / 'tp_sl_optimization_results.json'
best_json_path = OUTPUT_DIR / 'tp_sl_best_candidate.json'

results_df_export = results_df.copy()
if 'equity_curve' in results_df_export.columns:
    results_df_export['equity_curve'] = results_df_export['equity_curve'].apply(lambda x: x if isinstance(x, list) else [])

results_df_export.to_csv(results_csv_path, index=False)
results_df_export.to_json(results_json_path, orient='records', indent=2)

best_payload = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'selected_candidate': {
        'tp_mult': float(best_candidate['tp_mult']),
        'sl_mult': float(best_candidate['sl_mult']),
        'score': float(best_candidate['score']),
        'net_usd': float(best_candidate['net_usd']),
        'sharpe_ratio': float(best_candidate['sharpe_ratio']),
        'max_drawdown': float(best_candidate['max_drawdown']),
        'trade_count': int(best_candidate['trade_count']),
        'win_rate': float(best_candidate['win_rate']),
        'avg_trade_pnl': float(best_candidate['avg_trade_pnl']),
        'pack_path': str(best_candidate['pack_path']),
        'search_method': str(best_candidate['search_method']),
    },
    'config_snapshot': {
        'DS_NAME': DS_NAME,
        'N_ROWS': N_ROWS,
        'TRAIN_FRACTION': TRAIN_FRACTION,
        'TP_RANGE': TP_RANGE,
        'SL_RANGE': SL_RANGE,
        'N_BAYES_TRIALS': N_BAYES_TRIALS,
        'GRID_TP_VALUES': GRID_TP_VALUES,
        'GRID_SL_VALUES': GRID_SL_VALUES,
        'ALPHA_SHARPE': ALPHA_SHARPE,
        'MIN_TRADES_FOR_VALID_SCORE': min_trades_for_valid_score,
        'VALIDATION_MODE': VALIDATION_MODE,
        'OPTIMIZE_ONLY_TP_SL': OPTIMIZE_ONLY_TP_SL,
    },
}

with open(best_json_path, 'w', encoding='utf-8') as f:
    json.dump(best_payload, f, indent=2)

print('Saved:', results_csv_path)
print('Saved:', results_json_path)
print('Saved:', best_json_path)
print('Done. Best candidate:', best_payload['selected_candidate'])